# M1 — Semantic Scholar references (Colab + Drive)

Chạy **sau** notebook crawl arXiv của người 1 (`M1_Colab_Concurrent_Limited.ipynb`).
Quy trình: GitHub branch → Drive → đọc ID từ checkpoint `batch_*.tar.gz` → crawl references (resume) → xuất 2 bảng `papers` / `references` + validate.

- **1 request / publication** (S2 không phân version). Có `S2_API_KEY` thì ổn định hơn: thêm **Secret** tên `S2_API_KEY` trong Colab (biểu tượng chìa khóa). **Không dán key vào notebook.**
- Mỗi paper có trạng thái rõ ràng: `ok` / `not_indexed` (S2 chưa ghi nhận, 404) / `failed` (429, 5xx, mất mạng). Bài thật sự **0 reference** vẫn được ghi file nên không bị gọi lại.
- Resume: `ok` và `not_indexed` bị bỏ qua; `failed` được thử lại ở lần chạy sau. Muốn thử lại cả `not_indexed` (S2 index chậm 1–7 ngày) → `RETRY_NOT_INDEXED = True`.
- Lưu **toàn bộ** reference (kể cả không có arXiv ID); bảng `references` chỉ giữ ref có arXiv ID.
- Lỗi từng paper không dừng cả run.

**Cần push code Python mới nhất lên nhánh của nhóm trước khi chạy notebook.**

## SET UP — Clone code từ branch GitHub và cài thư viện

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/tuandatedu/arXir_project.git"  # TODO: thay URL repo nhóm
BRANCH = "M1-new"                    # TODO: tên branch thật
PROJECT = Path("/content/project")
assert "OWNER" not in REPO_URL, "Bạn chưa thay URL GitHub"

if not (PROJECT / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(PROJECT)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--ff-only", "origin", BRANCH], check=True)

os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_m1_pipeline.py"], check=True)
print("Repo:", PROJECT, "| Branch:", BRANCH)

## SET UP — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1 — Trỏ đúng thư mục Drive của range

`RANGE_ID` và đường dẫn `DATA_DIR` phải **trùng với notebook crawl của người 1** để tìm được checkpoint. Dữ liệu references được ghi xen vào cạnh đó:

```text
DATA_DIR/
  batch_00000.tar.gz + .json     # checkpoint của người 1 (nguyên trạng)
  references/<folder>/references_s2.json   # MỌI ref đã crawl (resume tại đây)
  tables/papers.json, references.json, metrics_s2_references.json, s2_crawl_records.csv
```

In [ ]:
RANGE_ID = 4

In [ ]:
import os
from pathlib import Path

# 1. DATA_DIR: checkpoint của Người 1 (chỉ đọc)
DATA_DIR = Path(
    f"/content/drive/MyDrive/arxiv-project-data/m1/range_{RANGE_ID}_dev"
)

# 2. Thư mục riêng trên Drive để lưu kết quả S2
MY_REFS_DIR = Path(
    f"/content/drive/MyDrive/references/range_{RANGE_ID}"
)

# reference/: lưu JSON riêng cho từng paper
# tables/: lưu papers, references và metrics
REFS_ROOT = MY_REFS_DIR / "reference"
TABLES_DIR = MY_REFS_DIR / "tables"

for path in (MY_REFS_DIR, REFS_ROOT, TABLES_DIR):
    path.mkdir(parents=True, exist_ok=True)

# 3. Kiểm tra checkpoint của Người 1
manifests = sorted(DATA_DIR.glob("batch_*.json"))

assert manifests, (
    f"Chưa tìm thấy batch_*.json trong {DATA_DIR} "
    "— kiểm tra lại lối tắt Drive"
)

print(f"Range: {RANGE_ID}")
print(f"Đọc checkpoint từ: {DATA_DIR}")
print(f"Số batch tìm thấy: {len(manifests)}")
print(f"Lưu references tại: {REFS_ROOT}")
print(f"Lưu tables tại: {TABLES_DIR}")

# 4. API key được gán trực tiếp
# Thay thế chuỗi dưới đây bằng API key Semantic Scholar thật của bạn
S2_API_KEY_DIRECT = "s2k-2GSKU1aa1guduuqNUaFQZSXLrZ5K5RwCGbPNRBN8"

if S2_API_KEY_DIRECT and S2_API_KEY_DIRECT != "YOUR_ACTUAL_S2_API_KEY_HERE":
    os.environ["S2_API_KEY"] = S2_API_KEY_DIRECT
    print("Đã nạp S2_API_KEY trực tiếp từ code.")
else:
    os.environ.pop("S2_API_KEY", None)
    print("Chưa cấu hình API Key. Sẽ chạy không key (chậm hơn, dễ bị lỗi 429).")

# 5. Cấu hình đường dẫn cho pipeline và metrics
from src.common import config
import src.common.metrics as metrics_module

config.WORKING_DIR = MY_REFS_DIR
config.TABLES_DIR = TABLES_DIR
metrics_module.TABLES_DIR = TABLES_DIR

# False: không tự động thử lại các paper từng được xác định là chưa index
RETRY_NOT_INDEXED = False

print("\nTABLES_DIR =", config.TABLES_DIR)
print("REFS_ROOT =", REFS_ROOT)

## 2 — Đọc danh sách paper từ checkpoint `batch_*.tar.gz`

Mỗi archive chứa `papers.jsonl` (metadata người 1 đã crawl). Lấy metadata của **mọi** bài, không lọc theo trạng thái tải TeX: bài chỉ có PDF vẫn là một hàng của bảng `papers` và vẫn có references. Bài S2/arXiv không tìm thấy (`not_found`) sẽ chỉ được báo cáo, không làm dừng notebook.

In [ ]:
import json
from pathlib import Path
import tarfile

from src.common.ids import normalize_arxiv_id
from src.m1_crawling.assigned_ranges import assigned_ids

# Đảm bảo các biến đường dẫn đã được khai báo trước đó (DATA_DIR, TABLES_DIR, RANGE_ID)
OUTPUT_TABLES_DIR = TABLES_DIR
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)


def _read_jsonl_from_archive(archive_path: Path, filename: str) -> list[dict]:
    """Đọc một file jsonl trong archive .tar.gz, không giải nén ra đĩa."""
    records = []
    with tarfile.open(archive_path, "r:gz") as tar:
        try:
            member = tar.extractfile(filename)
        except KeyError:
            raise ValueError(f"{archive_path.name} thiếu {filename}")
        if member:
            for line in member:
                line = line.strip()
                if line:
                    records.append(json.loads(line.decode("utf-8")))
    return records


def load_all_papers_from_all_batches() -> list[dict]:
    papers_dict = {}
    manifests = sorted(DATA_DIR.glob("batch_*.json"))
    print(f"Range {RANGE_ID} | Tìm thấy {len(manifests)} batch trong {DATA_DIR}")
    for index, manifest in enumerate(manifests, start=1):
        archive = manifest.with_suffix(".tar.gz")
        if not archive.exists():
            print(f"[warn] Bỏ qua (thiếu .tar.gz): {archive.name}")
            continue
        try:
            rows = _read_jsonl_from_archive(archive, "papers.jsonl")
        except (tarfile.TarError, EOFError, OSError, ValueError) as exc:
            print(f"[warn] Batch lỗi, cần crawl lại: {archive.name} ({exc})")
            continue
        for row in rows:
            try:
                paper_id = normalize_arxiv_id(row.get("arxiv_id") or "")
            except ValueError:
                print(
                    f"[warn] Bỏ qua dòng có ID không hợp lệ: {row.get('arxiv_id')!r}"
                )
                continue
            papers_dict[paper_id] = {
                "arxiv_id": paper_id,
                "title": row.get("title", ""),
                "authors": row.get("authors", []),
                "submission_date": row.get("submission_date", ""),
                "revised_dates": row.get("revised_dates", []),
                "venue": row.get("venue", None),
            }
        if index % 10 == 0 or index == len(manifests):
            print(
                f"  -> {index}/{len(manifests)} batch (tích lũy {len(papers_dict)} bài)"
            )
    return sorted(papers_dict.values(), key=lambda p: p["arxiv_id"])


def save_papers_to_json(papers_meta: list[dict], output_dir: Path):
    """Lưu danh sách metadata bài báo ra file papers.json."""
    if not papers_meta:
        print("[warn] Không có dữ liệu để lưu vào file JSON.")
        return

    json_path = output_dir / "papers.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(papers_meta, f, ensure_ascii=False, indent=2)
    print(f"✓ Đã lưu file JSON: {json_path}")


# --- Thực thi ---
papers_meta = load_all_papers_from_all_batches()
print(
    f"\nHOÀN THÀNH RANGE {RANGE_ID}: {len(papers_meta)} bài báo có metadata."
)

# Đối chiếu với danh sách ID đề giao (chỉ báo cáo, không assert)
expected = set(assigned_ids(RANGE_ID))
got = {p["arxiv_id"] for p in papers_meta}
missing = sorted(expected - got)
print(
    f"Đề giao: {len(expected)} | Có metadata: {len(got & expected)} | "
    f"Thiếu: {len(missing)} | Thừa ngoài range: {len(got - expected)}"
)
if missing:
    print(
        "[warn] Chưa có metadata (batch chưa crawl, lỗi, hoặc ID không tồn tại trên arXiv):",
        missing[:10],
        "..." if len(missing) > 10 else "",
    )
    print(
        "      Xem reports/crawl_records.csv của người 1 để biết status (error / not_found)."
    )

# Lưu dữ liệu ra file papers.json
save_papers_to_json(papers_meta, OUTPUT_TABLES_DIR)

## 3 — Crawl references (resume, có đo runtime)

Mỗi paper được lưu vào `REFS_ROOT/<folder>/`: `references_s2.json` (mọi reference, mỗi dòng một JSON) và `s2_status.json` (trạng thái, số reference, thời gian, lỗi). Ghi nguyên tử (ghi tạm rồi đổi tên), nên session Colab bị ngắt giữa chừng cũng không để lại file dở. Chạy lại cell này để tiếp tục.

In [ ]:
from collections import Counter
import csv
import json
import logging
import os
from pathlib import Path
import time
from typing import Any, Optional

from src.common.ids import normalize_arxiv_id
from src.common.io import write_json
from src.common.metrics import track_run
from src.m1_crawling.semantic_scholar import (
    _resolve_api_key,
    s2_ref_to_local_record,
)

logging.basicConfig(
    level=logging.INFO, format="%(levelname)s %(name)s: %(message)s"
)


def _s2_request_with_status(
    arxiv_id: str, api_key: Optional[str]
) -> tuple[Optional[dict[str, Any]], str, str]:
    """Gọi S2 API và phân loại phản hồi chi tiết.

    - ok: Request thành công (200 OK).
    - not_indexed: 404 / 400 (S2 chưa ghi nhận hoặc ID không tồn tại).
    - failed: Lỗi mạng, HTTP 429, 5xx.
    """
    from src.common.config import S2_FIELDS, S2_PAPER_URL
    from src.common.rate_limit import request_with_retry

    url = S2_PAPER_URL.format(arxiv_id=arxiv_id)
    headers: dict[str, str] = {}
    key = _resolve_api_key(api_key)
    if key:
        headers["x-api-key"] = key
    params = {"fields": ",".join(S2_FIELDS)}

    try:
        response = request_with_retry(
            "GET",
            url,
            headers=headers or None,
            params=params,
            timeout=60.0,
            max_retries=5,
        )
    except Exception as exc:
        return None, "failed", str(exc)

    status_code = getattr(response, "status_code", None)
    if status_code in (404, 400):
        return None, "not_indexed", f"HTTP {status_code}"
    if status_code is None or status_code >= 400:
        return None, "failed", f"HTTP {status_code}"

    try:
        payload = response.json()
        if isinstance(payload, dict):
            return payload, "ok", ""
        return None, "failed", "Invalid JSON structure"
    except Exception as exc:
        return None, "failed", f"JSON decode error: {exc}"


def run_s2_batch_with_metrics(
    papers_meta: list[dict[str, Any]],
    refs_root: Path,
    api_key: Optional[str] = None,
) -> tuple[list[dict[str, Any]], int]:
    results = []
    n_invalid_ids = 0
    refs_root.mkdir(parents=True, exist_ok=True)

    for index, paper in enumerate(papers_meta, start=1):
        raw_id = paper.get("arxiv_id", "")

        # 1. Kiểm tra arXiv ID không hợp lệ / thiếu (n_invalid_ids)
        try:
            if not raw_id:
                raise ValueError("Thiếu arxiv_id")
            base_id = normalize_arxiv_id(raw_id)
        except ValueError:
            n_invalid_ids += 1
            results.append({
                "arxiv_id": raw_id or "INVALID",
                "status": "invalid_id",
                "n_refs": 0,
                "n_arxiv": 0,
                "is_zero_ref": False,
                "is_api_error": True,
                "is_invalid_id": True,
                "from_cache": False,
                "elapsed_s": 0.0,
                "error": "arXiv ID không hợp lệ hoặc bị thiếu",
            })
            continue

        safe_filename = f"{base_id.replace('/', '-')}.json"
        refs_file = refs_root / safe_filename

        # 2. RESUME / CACHE CHECK (n_cache_hits)
        if refs_file.exists():
            try:
                with open(refs_file, "r", encoding="utf-8") as f:
                    records = json.load(f)

                n_refs = len(records)
                n_arxiv = sum(
                    1 for r in records if r.get("ref_arxiv_id") is not None
                )
                results.append({
                    "arxiv_id": base_id,
                    "status": "ok",
                    "n_refs": n_refs,
                    "n_arxiv": n_arxiv,
                    "is_zero_ref": (n_refs == 0),
                    "is_api_error": False,
                    "is_invalid_id": False,
                    "from_cache": True,
                    "elapsed_s": 0.0,
                    "error": "",
                })
                continue
            except Exception:
                pass  # File hỏng -> crawl lại

        # 3. CRAWL MỚI TỪ API
        start_time = time.perf_counter()
        payload, status, error_msg = _s2_request_with_status(base_id, api_key)
        elapsed = time.perf_counter() - start_time

        local_records = []
        if status == "ok" and payload is not None:
            raw_refs = payload.get("references") or []
            for item in raw_refs:
                if isinstance(item, dict):
                    local_records.append(s2_ref_to_local_record(base_id, item))

            # Chỉ lưu file khi API trả về hợp lệ (kể cả khi local_records rỗng)
            with open(refs_file, "w", encoding="utf-8") as f:
                json.dump(local_records, f, ensure_ascii=False, indent=2)

        n_arxiv = sum(
            1 for r in local_records if r.get("ref_arxiv_id") is not None
        )

        is_zero_ref = status == "ok" and len(local_records) == 0
        is_api_error = status == "failed"

        results.append({
            "arxiv_id": base_id,
            "status": status,
            "n_refs": len(local_records),
            "n_arxiv": n_arxiv,
            "is_zero_ref": is_zero_ref,
            "is_api_error": is_api_error,
            "is_invalid_id": False,
            "from_cache": False,
            "elapsed_s": elapsed,
            "error": error_msg,
        })

        if index % 50 == 0 or index == len(papers_meta):
            print(
                f"  -> Progress: {index}/{len(papers_meta)} papers processed..."
            )

    return results, n_invalid_ids


# --- THỰC THI VÀ TÍNH METRICS ---
if "papers_meta" not in globals() or not papers_meta:
    raise RuntimeError(
        "'papers_meta' chưa được tạo! Hãy chạy lại cell ở Mục 2 trước."
    )

started_at = time.perf_counter()

with track_run("s2_references") as metrics:
    metrics["range_id"] = RANGE_ID
    metrics["n_ids"] = len(papers_meta)

    results, n_invalid_ids = run_s2_batch_with_metrics(
        papers_meta=papers_meta,
        refs_root=REFS_ROOT,
        api_key=os.environ.get("S2_API_KEY"),
    )

    # Phân loại và tính 4 chỉ số yêu cầu
    ok_rows = [r for r in results if r["status"] == "ok"]
    failed_rows = [r for r in results if r["status"] == "failed"]
    cached_rows = [r for r in results if r.get("from_cache", False)]
    fresh_rows = [r for r in results if not r.get("from_cache", False)]

    n_zero_refs = sum(1 for r in results if r.get("is_zero_ref"))
    n_api_errors = sum(1 for r in results if r.get("is_api_error"))
    n_cache_hits = len(cached_rows)

    n_refs_total = sum(r.get("n_refs", 0) for r in ok_rows)
    n_refs_with_arxiv = sum(r.get("n_arxiv", 0) for r in ok_rows)

    fetched_times = [float(r.get("elapsed_s", 0) or 0) for r in fresh_rows]
    total_fetch_elapsed = sum(fetched_times)
    avg_fetch_time = (
        total_fetch_elapsed / len(fetched_times) if fetched_times else 0.0
    )

    cell_elapsed = time.perf_counter() - started_at

    # Lưu đầy đủ vào dictionary metrics
    metrics["s2_stats"] = {
        "n_invalid_ids": n_invalid_ids,
        "n_zero_refs": n_zero_refs,
        "n_api_errors": n_api_errors,
        "n_cache_hits": n_cache_hits,
        "n_papers_total": len(results),
        "n_ok": len(ok_rows),
        "n_not_indexed": sum(
            1 for r in results if r["status"] == "not_indexed"
        ),
        "n_refs_total": n_refs_total,
        "n_refs_with_arxiv": n_refs_with_arxiv,
        "avg_s2_seconds_per_fetched_paper": avg_fetch_time,
        "total_fetch_elapsed_sec": total_fetch_elapsed,
        "cell_elapsed_sec": cell_elapsed,
        "status_counts": dict(Counter(r["status"] for r in results)),
    }

# Ghi file metrics JSON
write_json(TABLES_DIR / "metrics_s2_references.json", metrics)

# Ghi bảng CSV chứa đầy đủ các thuộc tính chỉ số từng dòng
records_csv = TABLES_DIR / "s2_crawl_records.csv"
fields = [
    "arxiv_id",
    "status",
    "n_refs",
    "n_arxiv",
    "is_zero_ref",
    "is_api_error",
    "is_invalid_id",
    "from_cache",
    "elapsed_s",
    "error",
]

with records_csv.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    for record in results:
        writer.writerow({key: record.get(key, "") for key in fields})

print("\n✓ ĐÃ THÊM CÁC CỘT CHỈ SỐ VÀO BẢNG s2_crawl_records.csv:")
print(f" - n_invalid_ids : {n_invalid_ids}")
print(f" - n_zero_refs   : {n_zero_refs}")
print(f" - n_api_errors  : {n_api_errors}")
print(f" - n_cache_hits  : {n_cache_hits}")
print(f" - Đã lưu tại    : {records_csv}")

In [ ]:
# Ghi log từng paper (CSV) để đối chiếu / gửi người xử lý dữ liệu.
# `results` gồm mọi paper (kể cả lấy từ lần chạy trước), nên file này luôn đầy đủ.
import csv

records_csv = TABLES_DIR / "s2_crawl_records.csv"
fields = ["arxiv_id", "status", "n_refs", "n_arxiv", "from_cache", "elapsed_s", "error"]
with records_csv.open("w", newline="", encoding="utf-8-sig") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    for record in results:
        writer.writerow({key: record.get(key, "") for key in fields})
print("Đã ghi", records_csv)


## 4 — Recovery: chạy lại phần references dở dang

Chỉ chạy cell crawl ở mục 3 thêm lần nữa là đủ: paper đã lưu sẽ `skipped`, paper lỗi/chưa tới sẽ được thử lại.
Nếu 429 dồn dập thì chờ ≥ 5 phút rồi chạy lại; **không tăng tốc hoặc mở song song**.

## 5 — Xuất 2 bảng `papers` / `references` + validate

- `papers`: metadata đọc từ checkpoint (1 hàng / publication).
- `references`: chỉ ref **có arXiv ID** (matching pool của M3); mọi ref vẫn nằm trong `references/<folder>/references_s2.json`.
Kết quả validate (thiếu thông tin / sai kiểu / trùng lặp) được in ra và ghi vào metrics.

In [ ]:
import json
from pathlib import Path

from src.common.schemas import PaperRow, ReferenceRow
from src.m1_crawling.tables import (
    dedupe_reference_rows,
    s2_records_to_reference_rows,
    validate_m1_tables,
    write_m1_tables,
)

# Đảm bảo trỏ đường dẫn xuất file về đúng Drive
config.WORKING_DIR = MY_REFS_DIR
config.TABLES_DIR = TABLES_DIR

paper_rows: list[PaperRow] = []
reference_rows: list[ReferenceRow] = []

# 1. Gom dữ liệu papers và references từ các file JSON phẳng
for row in papers_meta:  # papers_meta đã unique theo arxiv_id
    base = row["arxiv_id"]

    # Tạo đối tượng PaperRow
    paper_rows.append(
        PaperRow(
            arxiv_id=base,
            title=row.get("title", ""),
            authors=row.get("authors") or [],
            submission_date=row.get("submission_date", ""),
            revised_dates=row.get("revised_dates") or [],
            venue=row.get("venue"),
        )
    )

    # Trỏ trực tiếp đến file JSON phẳng (ví dụ: 2307-00001.json)
    safe_filename = f"{base.replace('/', '-')}.json"
    target = REFS_ROOT / safe_filename

    if target.exists():
        try:
            with open(target, "r", encoding="utf-8") as f:
                raw_records = json.load(f)
            # Chuyển đổi các record hợp lệ vào reference_rows
            reference_rows.extend(
                s2_records_to_reference_rows(base, raw_records)
            )
        except Exception as exc:
            print(f"[warn] Không thể đọc file references {target}: {exc}")

# 2. Loại bỏ trùng lặp cặp (arxiv_id, ref_arxiv_id)
n_before = len(reference_rows)
reference_rows = dedupe_reference_rows(reference_rows)
print(
    f"references: {n_before} -> {len(reference_rows)} sau khi bỏ trùng cặp (arxiv_id, ref_arxiv_id)"
)

# 3. Validate dữ liệu 2 bảng
report = validate_m1_tables(paper_rows, reference_rows)
print(
    "validate ok =",
    report["ok"],
    "| papers =",
    report["n_papers"],
    "| references =",
    report["n_references"],
    "| errors =",
    report["n_errors"],
    "| warnings =",
    report["n_warnings"],
)

# Hiển thị các lỗi / cảnh báo nếu có
for severity in ("error", "warning"):
    shown = [i for i in report["issues"] if i["severity"] == severity][:10]
    for issue in shown:
        print(f"  [{severity}]", issue)

# 4. Ghi dữ liệu ra file papers.json và references.json
write_m1_tables(paper_rows, reference_rows)
write_json(TABLES_DIR / "validation_report.json", report)

# 5. Thống kê kết quả xuất file trên Drive
n_ref_files = len(list(REFS_ROOT.glob("*.json")))
print(f"\nTổng số file JSON phẳng đã cào trong REFS_ROOT: {n_ref_files} bài")

sample = next(REFS_ROOT.glob("*.json"), None)
if sample:
    with open(sample, "r", encoding="utf-8") as f:
        sample_rows = json.load(f)
    with_arxiv = sum(1 for r in sample_rows if r.get("ref_arxiv_id"))
    print(
        f"Ví dụ mẫu ({sample.name}): {len(sample_rows)} refs, {with_arxiv} có arXiv ID"
    )
    print("Một record mẫu:", sample_rows[0] if sample_rows else None)

print("\nĐÃ XUẤT THÀNH CÔNG:")
for name in (
    "papers.json",
    "references.json",
    "s2_crawl_records.csv",
    "metrics_s2_references.json",
    "validation_report.json",
):
    path = TABLES_DIR / name
    print(
        f"- {name}: {path.stat().st_size / 1024:.1f} KiB"
        if path.exists()
        else f"- {name}: MISSING"
    )

In [ ]:
import json
from pathlib import Path

# 1. Đếm tổng số file JSON phẳng trong REFS_ROOT
n_ref_files = len(list(REFS_ROOT.glob("*.json")))
print(f"File JSON references đã lưu: {n_ref_files} paper")

# 2. Lấy thử 1 file mẫu để kiểm tra nội dung
sample = next(REFS_ROOT.glob("*.json"), None)
if sample:
    try:
        with open(sample, "r", encoding="utf-8") as f:
            rows = json.load(f)
        with_arxiv = sum(1 for r in rows if r.get("ref_arxiv_id"))
        print(
            f"Ví dụ ({sample.name}): {len(rows)} refs, {with_arxiv} có arXiv ID"
        )
        print("Một record:", rows[0] if rows else None)
    except Exception as e:
        print(f"Lỗi khi đọc file mẫu {sample.name}: {e}")

print("\nTrạng thái các file trong TABLES_DIR:")
for name in (
    "papers.json",
    "references.json",
    "s2_crawl_records.csv",
    "metrics_s2_references.json",
    "validation_report.json",
):
    path = TABLES_DIR / name
    print(
        f"- {name}: {path.stat().st_size / 1024:.1f} KiB"
        if path.exists()
        else f"- {name}: MISSING"
    )

## Sau khi chạy

- Gửi cho người xử lý dữ liệu: `tables/papers.json`, `tables/references.json`, `tables/s2_crawl_records.csv` và thư mục `references/` trên Drive.
- `references/` chứa **mọi** reference (gồm ref không có arXiv ID) — đừng xóa trước khi submit Hugging Face ở M3.
- Chạy đầy đủ cả pipeline (source + S2 + bảng) trên máy:
  `python -m src.m1_crawling.run_m1 --ids assigned_ids.txt`
- Report Jupyter Book đọc `tables/metrics_s2_references.json` + `data/tables/metrics_m1.json` để vẽ figure lúc compile (`book/m1_report.md`).